# ML Model Training

This notebook mirrors the **Model Training** guide at [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/).

Train a Logistic Regression classifier on the preprocessed employee attrition dataset, track the experiment with MLflow, and log the model artifact to the registry.

**What we'll cover:**
1. Load training data
2. Configure MLflow
3. Build a preprocessing pipeline
4. Train and log the experiment
5. Inspect results

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri('http://mlflow:5000')
mlflow.set_experiment('employee-attrition')

## Step 1: Load training data

In [ ]:
train_df = pd.read_csv('/workspace/pipeline-data/train.csv')
test_df  = pd.read_csv('/workspace/pipeline-data/test.csv')

X_train = train_df.drop(columns=['Attrition'])
y_train = train_df['Attrition']
X_test  = test_df.drop(columns=['Attrition'])
y_test  = test_df['Attrition']

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Attrition rate — Train: {y_train.mean():.1%}  Test: {y_test.mean():.1%}')

## Step 2: Configure MLflow

In [ ]:
# Confirm the MLflow server is reachable
client = mlflow.tracking.MlflowClient()
experiments = client.search_experiments()
print(f'MLflow server: {mlflow.get_tracking_uri()}')
print(f'Existing experiments: {[e.name for e in experiments]}')

## Step 3: Build a preprocessing pipeline

In [ ]:
def build_pipeline(C=1.0, solver='lbfgs', max_iter=1000):
    """Return a sklearn Pipeline with StandardScaler + LogisticRegression."""
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(
            C=C, solver=solver, max_iter=max_iter, class_weight='balanced', random_state=42
        )),
    ])

A `Pipeline` chains preprocessing and model steps into a single object. This prevents data leakage: the scaler's `fit` is called only on training data, then `transform` is applied to both train and test. Wrapping the pipeline in MLflow gives us reproducible experiments — every run records its hyperparameters alongside its metrics.

## Step 4: Train and log the experiment

In [ ]:
with mlflow.start_run(run_name='lr-baseline') as run:
    C, solver, max_iter = 1.0, 'lbfgs', 1000

    mlflow.log_param('C', C)
    mlflow.log_param('solver', solver)
    mlflow.log_param('max_iter', max_iter)
    mlflow.log_param('class_weight', 'balanced')

    pipeline = build_pipeline(C=C, solver=solver, max_iter=max_iter)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1',       f1)
    mlflow.log_metric('roc_auc',  auc)

    mlflow.sklearn.log_model(pipeline, 'model', registered_model_name='attrition-classifier')

    run_id = run.info.run_id
    print(f'Run ID:   {run_id}')
    print(f'Accuracy: {acc:.4f}')
    print(f'F1 score: {f1:.4f}')
    print(f'ROC-AUC:  {auc:.4f}')

Open the MLflow UI at [mlflow.learnmlops.ops4life.com](https://mlflow.learnmlops.ops4life.com) and navigate to the **employee-attrition** experiment to see the run. The logged model appears in the **Models** tab as `attrition-classifier`.

## Step 5: Inspect results

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Left']))

## Next Steps

- Tune hyperparameters with Optuna: `02-pipeline/ml-model-training/tune.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/)